LLM Workflow with Q n A

In [ ]:
!pip install langchain langgraph dotenv

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict, List
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv()  # Load environment variables from .env file

In [ ]:
# load the default LLM model
# model = ChatOpenAI()

model = ChatOpenAI(    
    base_url="http://localhost:12434/engines/v1",
    api_key="docker", 
    temperature=0, 
    model = "ai/smollm2:360M-Q4_K_M")


In [ ]:
# Step 2: Create State
class AgentState(TypedDict):
    question: str
    answer: str


In [ ]:
def llm_qa(state: AgentState) -> AgentState:
    # extracrt the question from the state and use the LLM to answer it
    question = state["question"]
    
    # form a prompt for the LLM to answer the question
    prompt = f"Answer the following question: {question}"

    # invoke the LLM model to get the answer
    answer = model.invoke(prompt).content

    # update the state with the answer
    state["answer"] = answer
    return state

In [ ]:
# Create a workflow graph
graph = StateGraph(AgentState)

# add nodes to the graph
graph.add_node("llm_qa", llm_qa)

# add edges to the graph
graph.add_edge(START, "llm_qa")
graph.add_edge("llm_qa", END)

# compile graph into a workflow
workflow = graph.compile()

In [ ]:
from IPython.display import Image
Image(workflow.get_graph().draw_mermaid_png())

In [ ]:
# invoke the workflow with an initial state
initial_state = {"question": "How far is moon form the earth?", "answer": ""}
final_state = workflow.invoke(initial_state)

print("Question:", initial_state["question"])
print("Answer:", final_state["answer"])